# 6.3 Peta Hasil Klasifikasi — Jendela Demo Sidoarjo–Surabaya (Folium)

Input: `jatim/raster/s2_demo_sidoarjo.tif` (fase `demo`) + `jatim/model/jatim_best.joblib` (6.2).
Output: `jatim/model/demo_classified.tif` + peta Folium interaktif (geser/zoom, basemap Google Satellite, toggle layer) di `jatim/folium_demo_klasifikasi.html`.

In [1]:
import os
import numpy as np

RASTER = "jatim/raster/s2_demo_sidoarjo.tif"
MODEL = "jatim/model/jatim_best.joblib"
OUT_TIF = "jatim/model/demo_classified.tif"
OUT_PNG = "jatim/gambar/demo_preview.png"
OUT_HTML = "jatim/folium_demo_klasifikasi.html"
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
NAMES = {1: "Sawah", 2: "Bangunan", 3: "Mangrove", 4: "Lahan hijau", 5: "Laut", 6: "Danau"}

DATA_OK = os.path.exists(RASTER) and os.path.exists(MODEL)
print("raster:", "ADA" if os.path.exists(RASTER) else "BELUM TERSEDIA (fase demo)")
print("model:", "ADA" if os.path.exists(MODEL) else "BELUM TERSEDIA (6.2)")

raster: ADA
model: ADA


## Klasifikasi per piksel (12 fitur sama seperti training)

In [2]:
import joblib
import rasterio

if not DATA_OK:
    print("STATUS: TUNGGU — peta dilewati.")
elif os.path.exists(OUT_TIF):
    print("tif sudah ada — klasifikasi dilewati:", OUT_TIF)
else:
    clf = joblib.load(MODEL)
    src = rasterio.open(RASTER)
    arr = src.read().astype(float)  # (6, h, w) = B02,B03,B04,B08,B11,B12
    nodata = src.nodata
    h, w = arr.shape[1], arr.shape[2]
    B02, B03, B04, B08, B11, B12 = arr
    with np.errstate(divide="ignore", invalid="ignore"):
        feats = np.stack([B02, B03, B04, B08, B11, B12,
                          (B08 - B04) / (B08 + B04),
                          (B03 - B08) / (B03 + B08),
                          (B11 - B08) / (B11 + B08),
                          (B08 - B11) / (B08 + B11),
                          2.5 * (B08 - B04) / (B08 + 6 * B04 - 7.5 * B02 + 1),
                          ((B08 - B04) / (B08 + B04 + 0.5)) * 1.5])
    flat = feats.reshape(12, -1).T
    ok = np.isfinite(flat).all(axis=1)
    if nodata is not None:
        ok &= ~(arr.reshape(6, -1).T == nodata).any(axis=1)
    out = np.full(flat.shape[0], 255, dtype=np.uint8)
    idx = np.where(ok)[0]
    CH = 400000  # prediksi per chunk (SVM lambat di jutaan piksel)
    for s0 in range(0, len(idx), CH):
        sl = idx[s0:s0 + CH]
        out[sl] = clf.predict(flat[sl]).astype(np.uint8)
        print(f"    ... {min(s0 + CH, len(idx))}/{len(idx)} piksel", flush=True)
    n_valid = int(ok.sum())
    print(f"piksel valid: {n_valid} dari {flat.shape[0]}")
    for cid in sorted(NAMES):
        print(f"  kelas {cid} {NAMES[cid]}: {float((out[ok] == cid).mean()) * 100:.1f}%")
    meta = src.meta.copy()
    meta.update(count=1, dtype="uint8", nodata=255)
    with rasterio.open(OUT_TIF, "w", **meta) as dst:
        dst.write(out.reshape(h, w), 1)
    print("tersimpan:", OUT_TIF)

tif sudah ada — klasifikasi dilewati: jatim/model/demo_classified.tif


## Peta Folium: Google Satellite + overlay hasil + poligon sampel

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import folium
import geopandas as gpd

if not DATA_OK:
    print("STATUS: TUNGGU — folium dilewati.")
else:
    import rasterio
    src = rasterio.open(OUT_TIF)
    cls = src.read(1)
    src.close()
    # Batas jendela demo (WGS84) = DEMO_EXTENT fase download; tanpa transform PROJ
    w, s, e, n = 112.60, -7.55, 113.00, -7.05
    cmap = ListedColormap(["#000000", "#2e7d32", "#e53935", "#004d40", "#9ccc65", "#1565c0", "#4dd0e1"])
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.ma.masked_equal(cls, 255), vmin=0, vmax=6, cmap=cmap, interpolation="nearest")
    ax.axis("off")
    fig.savefig(OUT_PNG, dpi=110, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    m = folium.Map(location=[(s + n) / 2, (w + e) / 2], zoom_start=11, tiles=None, control_scale=True)
    folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
                     attr="Google Satellite", name="Google Satellite", overlay=False, max_zoom=20).add_to(m)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap", overlay=False).add_to(m)
    folium.raster_layers.ImageOverlay(OUT_PNG, bounds=[[s, w], [n, e]], name="Hasil klasifikasi (6 kelas)", opacity=0.65).add_to(m)
    g = gpd.read_file("jatim/training_samples_jawa_timur.gpkg", layer="training_samples", bbox=(w, s, e, n))
    fg = folium.FeatureGroup(name=f"Poligon sampel ({len(g)})")
    for _, r in g.iterrows():
        gj = gpd.GeoSeries([r.geometry], crs="EPSG:4326").to_json()
        folium.GeoJson(gj, style_function=lambda _: {"color": "yellow", "weight": 1.5, "fillOpacity": 0},
                       tooltip=f"Kelas {r.class_id} {NAMES[r.class_id]} | {r.source_class}").add_to(fg)
    fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(OUT_HTML)
    print("tersimpan:", OUT_PNG, "+", OUT_HTML)
    m

tersimpan: jatim/gambar/demo_preview.png + jatim/folium_demo_klasifikasi.html


## Status

- ▶ **6.4**: aplikasi Streamlit (`streamlit_jatim.py`) memakai `jatim_best.joblib` — tebak kelas dari input nilai band + visualisasi.